# Retrieval (v2 — Zero-LLM Routing via Summary Similarity)
**What changed from v1:**
- ❌ Old: LLM picks chapter (call 1) → LLM picks section (call 2) → LLM re-ranks pages (call 3)
- ✅ New: `keyword_overlap(query, node.summary)` routes to chapter and section — **0 LLM calls**
- ✅ Re-ranking removed entirely — section pages fed directly to answer generation
- ✅ Total LLM calls per query: **0 here** (1 in answerGeneration.ipynb)

This works because treeBuilder v2 stores rich merged summaries on every node.

**Run order:** `pageMetadata.ipynb` → `treeBuilder.ipynb` → `retreivalPDF.ipynb`

In [7]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


## 1. Imports & Connection

In [8]:
import json
import re

from src.config.db import get_connection

conn = get_connection()


## 2. Set Document & Query
Change `document_id` and `query` as needed.

In [9]:
document_id = "DOC000001"
query = "What are financial vulnerabilities?"


## 3. Tree Node Helpers (same as treeBuilder — runs standalone)

In [10]:
class TreeNode:
    def __init__(self, data: dict, parent=None):
        self.data     = data
        self.parent   = parent
        self.children = []

    @property
    def level(self):    return self.data.get("level", 0)
    @property
    def title(self):    return self.data.get("title", "")
    @property
    def summary(self):  return self.data.get("summary", "")
    @property
    def keywords(self): return self.data.get("keywords", [])
    @property
    def path(self):     return self.data.get("path", "")
    @property
    def type(self):     return self.data.get("type", "root")

    def __repr__(self):
        return f"TreeNode(level={self.level}, path={self.path!r}, title={self.title!r})"


def build_tree_nodes(tree_dict: dict, parent=None) -> "TreeNode":
    node = TreeNode(tree_dict, parent)
    for child_dict in tree_dict.get("children", []):
        node.children.append(build_tree_nodes(child_dict, parent=node))
    return node


def load_tree_from_db(conn, document_id: str) -> "TreeNode":
    with conn.cursor() as cur:
        cur.execute(
            'SELECT "treeJson" FROM "Tree" WHERE "documentId" = %s',
            (document_id,),
        )
        row = cur.fetchone()

    if row is None:
        raise ValueError(f"No tree found for documentId={document_id!r}")

    tree_dict = row[0]
    if isinstance(tree_dict, str):
        tree_dict = json.loads(tree_dict)

    return build_tree_nodes(tree_dict)


## 4. Load the Tree

In [11]:
tree_root = load_tree_from_db(conn, document_id)

print("Root:", tree_root.title)
print(f"Root summary: {tree_root.summary[:150]}...")
print(f"Chapters: {len(tree_root.children)}")
for ch in tree_root.children:
    print(f"  [{ch.path}] {ch.title}  ({len(ch.children)} sections)")


Root: 2023-annual-report-truncated
Root summary: The 2023 annual report of the Federal Reserve System provides a comprehensive overview of its activities, including monetary policy operations and fin...
Chapters: 5
  [1] Introduction and Overview  (5 sections)
  [2] Monetary Policy and Economic Developments  (8 sections)
  [3] Financial Stability  (5 sections)
  [4] Supervision and Regulation  (8 sections)
  [5] International Engagement and Financial Stability  (4 sections)


## 5. Keyword Overlap Similarity (Zero LLM Calls)
Replaces both LLM traversal calls.
Scores each node by how many query words appear in its merged summary + keywords.
The node with the highest score wins.

In [12]:
def keyword_overlap(query: str, node: "TreeNode") -> float:
    """
    Score a tree node against the query using keyword overlap.
    Checks against: node summary + node keywords + node title.
    Returns overlap ratio — higher is more relevant.
    """
    query_words = set(re.sub(r"[^a-z0-9 ]", " ", query.lower()).split())

    # Build node text from summary + keywords + title (all rich after bottom-up merge)
    node_text = " ".join([
        node.summary,
        " ".join(node.keywords),
        node.title,
    ]).lower()
    node_words = set(re.sub(r"[^a-z0-9 ]", " ", node_text).split())

    overlap = query_words & node_words
    # Normalise by query length so short queries aren't penalised
    return len(overlap) / (len(query_words) + 1e-9)


def find_best_chapter(query: str, root: "TreeNode") -> "TreeNode":
    """Pick chapter with highest keyword overlap against the query."""
    return max(root.children, key=lambda ch: keyword_overlap(query, ch))


def find_best_section(query: str, chapter: "TreeNode") -> "TreeNode":
    """Pick section within a chapter with highest keyword overlap."""
    return max(chapter.children, key=lambda sec: keyword_overlap(query, sec))


def traverse_tree(query: str, root: "TreeNode") -> dict:
    """
    Route query to best chapter → best section using summary similarity only.
    Zero LLM calls. Returns same dict shape as v1 for compatibility.
    """
    best_chapter = find_best_chapter(query, root)
    best_section = find_best_section(query, best_chapter)

    path = [root, best_chapter, best_section]

    return {"path": path, "leaf": best_section}


## 6. Run Traversal (0 LLM calls)

In [13]:
traversal = traverse_tree(query, tree_root)
leaf = traversal["leaf"]

print("Traversal path:")
for node in traversal["path"]:
    score = keyword_overlap(query, node)
    print(f"  [{node.type}] {node.title}  (path={node.path}, score={score:.3f})")

print()
print(f"Leaf section: '{leaf.title}'  pages {leaf.data.get('pageStart')}-{leaf.data.get('pageEnd')}")
print(f"pageIds in leaf: {len(leaf.data.get('pageIds', []))}")


Traversal path:
  [root] 2023-annual-report-truncated  (path=root, score=0.500)
  [chapter] Financial Stability  (path=3, score=0.500)
  [section] Introduction to Financial Stability  (path=3.1, score=0.500)

Leaf section: 'Introduction to Financial Stability'  pages 21-22
pageIds in leaf: 2


## 7. Fetch Pages from Winning Section (no re-ranking)
Re-ranking removed — the section already has 2-4 highly relevant pages.
All pages in the section are passed directly to answer generation.

In [14]:
def get_candidate_pages(conn, leaf: "TreeNode") -> list:
    """Fetch pageNumber + metadata for every page in the leaf section."""
    page_ids = leaf.data.get("pageIds", [])
    if not page_ids:
        return []

    placeholders = ",".join(["%s"] * len(page_ids))
    with conn.cursor() as cur:
        cur.execute(
            f'''
            SELECT "pageNumber", metadata
            FROM "Page"
            WHERE id IN ({placeholders})
            ORDER BY "pageNumber"
            ''',
            page_ids,
        )
        rows = cur.fetchall()

    candidates = []
    for page_number, metadata in rows:
        meta = metadata or {}
        candidates.append({
            "pageNumber": page_number,
            "title":      meta.get("title", ""),
            "summary":    meta.get("summary", ""),
            "keywords":   meta.get("keywords", []),
        })
    return candidates


def fetch_pages_content(conn, document_id: str, page_numbers: list) -> dict:
    """Fetch full page text for the given page numbers, keyed by pageNumber."""
    if not page_numbers:
        return {}

    placeholders = ",".join(["%s"] * len(page_numbers))
    with conn.cursor() as cur:
        cur.execute(
            f'''
            SELECT "pageNumber", content
            FROM "Page"
            WHERE "documentId" = %s
              AND "pageNumber" IN ({placeholders})
            ''',
            (document_id, *page_numbers),
        )
        rows = cur.fetchall()

    content_by_page = {page_number: content for page_number, content in rows}
    return {p: content_by_page[p] for p in page_numbers if p in content_by_page}


candidates    = get_candidate_pages(conn, leaf)
page_numbers  = [c["pageNumber"] for c in candidates]
pages_content = fetch_pages_content(conn, document_id, page_numbers)

print(f"Fetched {len(candidates)} candidate pages (no re-ranking needed)")
for c in candidates:
    print(f"  Page {c['pageNumber']}: {c['title']}")


Fetched 2 candidate pages (no re-ranking needed)
  Page 21: Financial Stability
  Page 22: Monitoring Financial Vulnerabilities


## 8. Full Retrieval Pipeline
Same return shape as v1 — `answerGeneration.ipynb` calls this without changes.
`ranked_pages` and `top_pages` still present for compatibility.

In [15]:
def retrieve(query: str, document_id: str, conn, top_k: int = 5) -> dict:
    """
    Zero-LLM retrieval: summary similarity routing → fetch all section pages.

    Returns same dict shape as v1 for full compatibility with answerGeneration.ipynb:
      tree_path      - traversal path (root → chapter → section)
      leaf           - winning section raw data
      candidates     - candidate page metadata
      ranked_pages   - page numbers in section order (no re-ranking)
      top_pages      - top_k page numbers
      pages_content  - {pageNumber: full_text}
    """
    tree_root  = load_tree_from_db(conn, document_id)
    traversal  = traverse_tree(query, tree_root)
    leaf       = traversal["leaf"]

    candidates    = get_candidate_pages(conn, leaf)
    page_numbers  = [c["pageNumber"] for c in candidates]
    top_pages     = page_numbers[:top_k]
    pages_content = fetch_pages_content(conn, document_id, top_pages)

    return {
        "tree_path": [
            {"type": n.type, "title": n.title, "path": n.path}
            for n in traversal["path"]
        ],
        "leaf":          leaf.data,
        "candidates":    candidates,
        "ranked_pages":  page_numbers,   # in section order, no LLM re-rank
        "top_pages":     top_pages,
        "pages_content": pages_content,
    }


## 9. Demo

In [16]:
result = retrieve(query, document_id, conn, top_k=5)

print("Query:", query)
print()
print("Tree path:")
for node in result["tree_path"]:
    print(f"  [{node['type']}] {node['title']}")
print()
print("Top pages used for answer generation:", result["top_pages"])
print()
print("LLM calls made during retrieval: 0")


Query: What are financial vulnerabilities?

Tree path:
  [root] 2023-annual-report-truncated
  [chapter] Financial Stability
  [section] Introduction to Financial Stability

Top pages used for answer generation: [21, 22]

LLM calls made during retrieval: 0
